In [85]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)


In [86]:
df=pd.read_csv("/content/ecommerce_fraud_transactions_1.csv")

In [87]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 17 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   transaction_id                      200000 non-null  int64  
 1   transaction_amount                  200000 non-null  float64
 2   transaction_hour                    200000 non-null  int64  
 3   customer_age                        200000 non-null  int64  
 4   account_age_days                    200000 non-null  int64  
 5   num_previous_transactions           200000 non-null  int64  
 6   payment_method                      200000 non-null  object 
 7   device_type                         200000 non-null  object 
 8   distance_from_home_km               200000 non-null  float64
 9   merchant_category                   200000 non-null  object 
 10  num_items                           200000 non-null  int64  
 11  is_first_purchase         

In [88]:
df.head()

,transaction_id,transaction_amount,transaction_hour,customer_age,account_age_days,num_previous_transactions,payment_method,device_type,distance_from_home_km,merchant_category,num_items,is_first_purchase,shipping_billing_match,customer_gender,day_of_week,avg_transaction_value_last_30_days,is_fraud
0,1,49.24,12,33,1822,10,debit_card,desktop,2.72,groceries,1,0,1,M,Tuesday,36.44,0
1,2,26.09,10,22,558,12,paypal,desktop,11.76,toys,1,0,1,F,Wednesday,25.75,0
2,3,57.26,4,50,2967,18,credit_card,desktop,100.06,home,2,0,1,F,Monday,66.65,1
3,4,137.42,7,26,53,13,paypal,mobile,5.67,digital_goods,2,0,1,Other,Tuesday,137.07,0
4,5,23.71,22,45,3385,15,credit_card,desktop,2.56,electronics,2,0,1,M,Saturday,23.68,0


In [89]:
df.isnull().sum()

,0
transaction_id,0
transaction_amount,0
transaction_hour,0
customer_age,0
account_age_days,0
num_previous_transactions,0
payment_method,0
device_type,0
distance_from_home_km,0
merchant_category,0


In [90]:
df["customer_gender"].unique()

array(['M', 'F', 'Other', nan], dtype=object)

In [91]:
X = df.drop("is_fraud", axis=1)

y = df["is_fraud"]

In [92]:
numerical_columns = X.select_dtypes(include=np.number).columns

categorical_columns = X.select_dtypes(include="object").columns


In [93]:
numerical_transformer = SimpleImputer(strategy="median")

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_columns),
        ("cat", categorical_transformer, categorical_columns)
    ]
)

In [94]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [95]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        ("regressor",
        RandomForestClassifier(
             n_estimators=100,
             random_state=42
         ))
    ]
)

In [96]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  SimpleImputer(strategy='median'),
                                                  Index(['transaction_id', 'transaction_amount', 'transaction_hour',
       'customer_age', 'account_age_days', 'num_previous_transactions',
       'distance_from_home_km', 'num_items', 'is_first_purchase',
       'shipping_billing_match', 'avg_transaction_value_last_30_days'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['payment_method', 'device_type', 'merchant_category', 'customer_gender',
       'day_of_week'],
      dtype='object'))])),
                ('regressor', RandomForestClassifier(random_state=42))])

In [97]:
df.head()

,transaction_id,transaction_amount,transaction_hour,customer_age,account_age_days,num_previous_transactions,payment_method,device_type,distance_from_home_km,merchant_category,num_items,is_first_purchase,shipping_billing_match,customer_gender,day_of_week,avg_transaction_value_last_30_days,is_fraud
0,1,49.24,12,33,1822,10,debit_card,desktop,2.72,groceries,1,0,1,M,Tuesday,36.44,0
1,2,26.09,10,22,558,12,paypal,desktop,11.76,toys,1,0,1,F,Wednesday,25.75,0
2,3,57.26,4,50,2967,18,credit_card,desktop,100.06,home,2,0,1,F,Monday,66.65,1
3,4,137.42,7,26,53,13,paypal,mobile,5.67,digital_goods,2,0,1,Other,Tuesday,137.07,0
4,5,23.71,22,45,3385,15,credit_card,desktop,2.56,electronics,2,0,1,M,Saturday,23.68,0


In [98]:
df.isnull().sum()

,0
transaction_id,0
transaction_amount,0
transaction_hour,0
customer_age,0
account_age_days,0
num_previous_transactions,0
payment_method,0
device_type,0
distance_from_home_km,0
merchant_category,0


In [99]:
y_pred = model.predict(X_test)

In [100]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)



Accuracy: 0.94885


In [101]:


new_data = pd.DataFrame({
    "transaction_id": [int(input("Transaction ID: "))],
    "transaction_amount": [float(input("Transaction Amount: "))],
    "transaction_hour": [int(input("Transaction Hour (0-23): "))],
    "customer_age": [int(input("Customer Age: "))],
    "account_age_days": [int(input("Account Age (days): "))],
    "num_previous_transactions": [int(input("Number of Previous Transactions: "))],
    "payment_method": [input("Payment Method (credit_card/debit_card/paypal): ")],
    "device_type": [input("Device Type (mobile/desktop/tablet): ")],
    "distance_from_home_km": [float(input("Distance from Home (km): "))],
    "merchant_category": [input("Merchant Category: ")],
    "num_items": [int(input("Number of Items: "))],
    "is_first_purchase": [int(input("Is First Purchase? (0=No, 1=Yes): "))],
    "shipping_billing_match": [int(input("Shipping & Billing Match? (0=No, 1=Yes): "))],
    "customer_gender": [input("Customer Gender (M/F): ")],
    "day_of_week": [input("Day of Week: ")],
    "avg_transaction_value_last_30_days": [float(input("Average Transaction Value (Last 30 Days): "))]
})

prediction = model.predict(new_data)

if prediction[0] == 1:
    print("\nPrediction: Fraud Transaction")
else:
    print("\nPrediction: Genuine Transaction")

Transaction ID: 2
Transaction Amount: 56
Transaction Hour (0-23): 3
Customer Age: 19
Account Age (days): 296
Number of Previous Transactions: 23
Payment Method (credit_card/debit_card/paypal): paypal
Device Type (mobile/desktop/tablet): mobile
Distance from Home (km): 2
Merchant Category: toys
Number of Items: 1
Is First Purchase? (0=No, 1=Yes): 0
Shipping & Billing Match? (0=No, 1=Yes): 1
Customer Gender (M/F): m
Day of Week: 5
Average Transaction Value (Last 30 Days): 20

Prediction: Genuine Transaction
